# Cell Tracking Challenge: WAFT vs Lucas-Kanade

This notebook downloads the **Fluo-C2DL-Huh7** 2D+time training dataset from the Cell Tracking Challenge and compares optical flow from **WAFT** and **Lucas-Kanade** on consecutive microscopy frames.

It mirrors the visualization style used in `deep_flow_demo.ipynb`: input frames, color-coded flow, quiver plots, and runtime summaries. Because this dataset does not provide dense optical-flow ground truth, the comparison is visual and timing-based rather than AAE/AEPE-based.


In [ ]:
from __future__ import annotations

import io
import os
import sys
import time
import zipfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Make local package imports work whether the notebook is launched from the
# repository root or from the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from optical_flow import estimate_flow, flow_to_color, plot_flow

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 100


## Dataset and Runtime Settings

The notebook stores Cell Tracking Challenge downloads under `data/cell_tracking_challenge/`, which is ignored by git. The default sequence is `01` from the Fluo-C2DL-Huh7 training set.

WAFT can be slow on CPU because it runs a deep network and may download model weights on first use. Use `FULL_SEQUENCE_MAX_PAIRS` to cap the full-sequence GIF pass while developing; set it to `None` to process the whole sequence.


In [ ]:
DATASET_NAME = "Fluo-C2DL-Huh7"
DATASET_URL = "https://data.celltrackingchallenge.net/training-datasets/Fluo-C2DL-Huh7.zip"
SEQUENCE_ID = "01"
PAIR_INDEX = 0

DATA_ROOT = PROJECT_ROOT / "data" / "cell_tracking_challenge"
ZIP_PATH = DATA_ROOT / f"{DATASET_NAME}.zip"
EXTRACT_DIR = DATA_ROOT / DATASET_NAME
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "ctc_waft_lk_comparison"

WAFT_PARAMS = {"model_name": "waft-sintel", "iters": 20}
LK_PARAMS = {"window_size": 15}
QUIVER_STEP = 12
GIF_DURATION_MS = 160
FULL_SEQUENCE_MAX_PAIRS = None  # Set to an integer for a quick preview run.

DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{
    "dataset": DATASET_NAME,
    "zip_path": str(ZIP_PATH),
    "extract_dir": str(EXTRACT_DIR),
    "output_dir": str(OUTPUT_DIR),
}


## Download and Extract the CTC Dataset

This cell downloads the official training zip only if it is not already present, then extracts it once into the ignored data directory.


In [ ]:
def download_with_progress(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)

    def report(block_num: int, block_size: int, total_size: int) -> None:
        if total_size <= 0:
            return
        downloaded = min(block_num * block_size, total_size)
        pct = 100 * downloaded / total_size
        print(f"\rDownloading {destination.name}: {pct:5.1f}%", end="")

    urllib.request.urlretrieve(url, destination, reporthook=report)
    print()


def ensure_dataset() -> Path:
    if ZIP_PATH.exists():
        print(f"Using existing zip: {ZIP_PATH}")
    else:
        print(f"Downloading {DATASET_NAME} from {DATASET_URL}")
        download_with_progress(DATASET_URL, ZIP_PATH)

    marker = EXTRACT_DIR / ".extract_complete"
    if marker.exists():
        print(f"Using existing extraction: {EXTRACT_DIR}")
        return EXTRACT_DIR

    print(f"Extracting {ZIP_PATH.name} to {DATA_ROOT}")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)

    if not EXTRACT_DIR.exists():
        matches = [p for p in DATA_ROOT.iterdir() if p.is_dir() and DATASET_NAME in p.name]
        if not matches:
            raise FileNotFoundError(f"Could not find extracted directory for {DATASET_NAME}")
        extracted = matches[0]
    else:
        extracted = EXTRACT_DIR

    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.touch()
    print(f"Extracted dataset: {extracted}")
    return extracted


dataset_dir = ensure_dataset()
dataset_dir


## Discover Frames and Preview the Selected Pair

The CTC training archive contains multiple numbered sequences. This section finds image directories, loads a consecutive pair, and normalizes microscopy intensities to `uint8` for both visualization and optical-flow input.


In [ ]:
def find_sequence_dirs(root: Path) -> dict[str, Path]:
    sequence_dirs = {}
    for path in sorted(root.rglob("*")):
        if not path.is_dir():
            continue
        if any(part.endswith(("_GT", "_ST")) for part in path.parts):
            continue
        frames = sorted(path.glob("*.tif")) + sorted(path.glob("*.tiff"))
        if len(frames) >= 2:
            sequence_dirs[path.name] = path
    return sequence_dirs


def list_frames(sequence_dir: Path) -> list[Path]:
    frames = sorted(sequence_dir.glob("*.tif")) + sorted(sequence_dir.glob("*.tiff"))
    if len(frames) < 2:
        raise ValueError(f"Need at least two frames in {sequence_dir}")
    return frames


def normalize_to_uint8(image: np.ndarray, lower_pct: float = 1.0, upper_pct: float = 99.5) -> np.ndarray:
    image = np.asarray(image, dtype=np.float32)
    lo, hi = np.percentile(image, [lower_pct, upper_pct])
    if hi <= lo:
        hi = float(image.max())
        lo = float(image.min())
    if hi <= lo:
        return np.zeros(image.shape, dtype=np.uint8)
    scaled = (image - lo) / (hi - lo)
    return (255 * np.clip(scaled, 0, 1)).astype(np.uint8)


def load_frame(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        arr = np.array(img)
    if arr.ndim == 3 and arr.shape[-1] > 1:
        arr = arr[..., 0]
    return normalize_to_uint8(arr)


sequence_dirs = find_sequence_dirs(dataset_dir)
sequence_dir = sequence_dirs.get(SEQUENCE_ID)
if sequence_dir is None:
    raise KeyError(f"Sequence {SEQUENCE_ID!r} not found. Available: {sorted(sequence_dirs)}")

frame_paths = list_frames(sequence_dir)
PAIR_INDEX = min(PAIR_INDEX, len(frame_paths) - 2)
im1 = load_frame(frame_paths[PAIR_INDEX]).astype(float)
im2 = load_frame(frame_paths[PAIR_INDEX + 1]).astype(float)

print(f"Sequence: {sequence_dir}")
print(f"Frames: {len(frame_paths)}")
print(f"Selected pair: {frame_paths[PAIR_INDEX].name} -> {frame_paths[PAIR_INDEX + 1].name}")
print(f"Image shape: {im1.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(im1, cmap="gray")
axes[0].set_title(frame_paths[PAIR_INDEX].name)
axes[0].axis("off")

axes[1].imshow(im2, cmap="gray")
axes[1].set_title(frame_paths[PAIR_INDEX + 1].name)
axes[1].axis("off")

plt.suptitle(f"{DATASET_NAME} sequence {SEQUENCE_ID}: consecutive frames")
plt.tight_layout()
plt.show()

## Single-Pair WAFT vs Lucas-Kanade

Run both algorithms on the selected sequential pair. The table reports wall-clock time and simple flow-magnitude summaries, since there is no dense ground-truth optical flow for these CTC frames.

In [ ]:
def run_flow_pair(name: str, method: str, params: dict, frame_a: np.ndarray, frame_b: np.ndarray) -> dict:
    print(f"Running {name}...")
    start = time.time()
    uv = estimate_flow(frame_a, frame_b, method=method, params=params)
    elapsed = time.time() - start
    magnitude = np.linalg.norm(uv, axis=2)
    result = {
        "name": name,
        "method": method,
        "params": params,
        "flow": uv,
        "time_s": elapsed,
        "mean_mag": float(np.mean(magnitude)),
        "median_mag": float(np.median(magnitude)),
        "p95_mag": float(np.percentile(magnitude, 95)),
    }
    print(
        f"  time={elapsed:.2f}s, mean|flow|={result['mean_mag']:.3f}px, "
        f"p95|flow|={result['p95_mag']:.3f}px"
    )
    return result


single_pair_results = [
    run_flow_pair("WAFT", "waft", WAFT_PARAMS, im1, im2),
    run_flow_pair("Lucas-Kanade", "lk", LK_PARAMS, im1, im2),
]

print(f"{'Method':<16} {'Time (s)':>10} {'Mean |flow|':>14} {'Median |flow|':>16} {'P95 |flow|':>12}")
print("-" * 74)
for result in single_pair_results:
    print(
        f"{result['name']:<16} {result['time_s']:>10.2f} "
        f"{result['mean_mag']:>14.3f} {result['median_mag']:>16.3f} {result['p95_mag']:>12.3f}"
    )

In [ ]:
fig, axes = plt.subplots(2, len(single_pair_results), figsize=(7 * len(single_pair_results), 9))
if len(single_pair_results) == 1:
    axes = axes[:, None]

for i, result in enumerate(single_pair_results):
    uv = result["flow"]
    axes[0, i].imshow(flow_to_color(uv))
    axes[0, i].set_title(
        f"{result['name']} color flow\n"
        f"time={result['time_s']:.2f}s, p95={result['p95_mag']:.2f}px"
    )
    axes[0, i].axis("off")

    plot_flow(uv, style="quiver", ax=axes[1, i], step=QUIVER_STEP)
    axes[1, i].set_title(f"{result['name']} quiver")

plt.suptitle(f"WAFT vs Lucas-Kanade on {DATASET_NAME} pair {PAIR_INDEX}", fontsize=14)
plt.tight_layout()
plt.show()

## Full-Sequence GIFs

For each method, compute flow between every consecutive frame pair in the selected sequence, then save a GIF with three synchronized panels: the actual microscopy frame, a quiver plot, and angular/color flow.

The helper accepts `max_pairs` so you can make quick preview GIFs before running the full sequence.

In [ ]:
def figure_to_pil(fig: plt.Figure) -> Image.Image:
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def render_flow_movie_frame(frame: np.ndarray, uv: np.ndarray, title: str, quiver_step: int) -> Image.Image:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(frame, cmap="gray")
    axes[0].set_title("Actual data")
    axes[0].axis("off")

    axes[1].imshow(frame, cmap="gray")
    h, w = uv.shape[:2]
    y, x = np.mgrid[0:h:quiver_step, 0:w:quiver_step]
    axes[1].quiver(
        x,
        y,
        uv[::quiver_step, ::quiver_step, 0],
        uv[::quiver_step, ::quiver_step, 1],
        color="yellow",
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.003,
    )
    axes[1].set_ylim(h, 0)
    axes[1].set_xlim(0, w)
    axes[1].set_aspect("equal")
    axes[1].set_title("Optical flow quiver")
    axes[1].axis("off")

    axes[2].imshow(flow_to_color(uv))
    axes[2].set_title("Angular/color flow")
    axes[2].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    return figure_to_pil(fig)


def save_flow_gif(
    method_label: str,
    method: str,
    params: dict,
    frames: list[Path],
    output_path: Path,
    max_pairs: int | None = None,
    quiver_step: int = QUIVER_STEP,
    duration_ms: int = GIF_DURATION_MS,
) -> Path:
    pair_count = len(frames) - 1 if max_pairs is None else min(max_pairs, len(frames) - 1)
    if pair_count <= 0:
        raise ValueError("Need at least one frame pair for GIF generation")

    rendered_frames = []
    for idx in range(pair_count):
        frame_a = load_frame(frames[idx]).astype(float)
        frame_b = load_frame(frames[idx + 1]).astype(float)
        print(f"{method_label}: pair {idx + 1}/{pair_count} ({frames[idx].name} -> {frames[idx + 1].name})")
        uv = estimate_flow(frame_a, frame_b, method=method, params=params)
        rendered_frames.append(
            render_flow_movie_frame(
                frame_a,
                uv,
                f"{method_label}: {frames[idx].name} -> {frames[idx + 1].name}",
                quiver_step=quiver_step,
            )
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    rendered_frames[0].save(
        output_path,
        save_all=True,
        append_images=rendered_frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=True,
    )
    print(f"Saved {output_path}")
    return output_path

In [ ]:
gif_jobs = [
    ("waft", "WAFT", "waft", WAFT_PARAMS),
    ("lucas_kanade", "Lucas-Kanade", "lk", LK_PARAMS),
]

gif_paths = {}
for slug, label, method, params in gif_jobs:
    gif_paths[label] = save_flow_gif(
        method_label=label,
        method=method,
        params=params,
        frames=frame_paths,
        output_path=OUTPUT_DIR / f"{DATASET_NAME}_{SEQUENCE_ID}_{slug}.gif",
        max_pairs=FULL_SEQUENCE_MAX_PAIRS,
        quiver_step=QUIVER_STEP,
        duration_ms=GIF_DURATION_MS,
    )

gif_paths

## Notes

- The CTC dataset is downloaded from the official 2D+time dataset page and saved under `data/cell_tracking_challenge/`.
- WAFT model weights are handled by the package model cache on first use.
- The GIF output directory is separate from the downloaded data so figures can be inspected without committing the dataset itself.